In [ ]:
import pandas as pd
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')


In [ ]:
import mlflow


In [4]:
# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")


<Experiment: artifact_location=('file:c:/Users/lara-/OneDrive/Desktop/POS TECH '
 'ML/projeto-tech-challenge-churn/notebooks/mlruns/1'), creation_time=1774800663369, experiment_id='1', last_update_time=1775007298338, lifecycle_stage='active', name='Projeto_Churn_Tech_Challenge', tags={}, workspace='default'>

# --------------- REGRESSÃO LOGISTICA ---------------

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.data
import matplotlib.pyplot as plt
import os
import json
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature

csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(
    df, source=csv_path, name="Churn_Analysis_Base")

# --- 1. CONFIGURAÇÕES E CUSTOS (Ticket Médio R$ 65) ---
test_size = 0.2
random_state = 42
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65,00
COST_FN = TICKET_MEDIO * 5    # R$ 325,00

# Cria a pasta de saída para os artefatos locais antes do upload
os.makedirs("outputs", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(),
         X_train.select_dtypes(include=['int64', 'float64']).columns)
    ]
)

# Fit e transformando para DataFrames (melhor para as signatures do MLflow)
X_train_df = pd.DataFrame(preprocessador.fit_transform(X_train),
                          columns=preprocessador.get_feature_names_out())
X_test_df = pd.DataFrame(preprocessador.transform(X_test),
                         columns=preprocessador.get_feature_names_out())

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="regressao_logistica_baseline_3") as run:

    model = LogisticRegression(random_state=random_state, max_iter=1000)
    mlflow.log_input(dataset, context="training")
    model.fit(X_train_df, y_train)

    # Predições e Probabilidades
    y_probs = model.predict_proba(X_test_df)[:, 1]
    threshold = 0.40  # Threshold ajustado para priorizar menor FN de Churn
    y_pred = (y_probs >= threshold).astype(int)

    # Calculando as métricas solicitadas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    cm = confusion_matrix(y_test, y_pred)
    total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

    # Log de Tags, Parâmetros e Metrics
    mlflow.set_tags({"phase": "baseline", "model_type": "logistic_regression"})

    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": random_state,
        "test_size": test_size,
        "stratify": True,
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "threshold": threshold,
        "ticket_medio": TICKET_MEDIO,
        "cost_fp": COST_FP,
        "cost_fn": COST_FN
    })

    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.total_cost": total_cost
    })

    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='darkorange',
                lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC (Receiver Operating Characteristic)")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate (Recall)")
    ax_roc.legend(loc="lower right")
    fig_roc.savefig("outputs/roc_curve.png")
    mlflow.log_artifact("outputs/roc_curve.png", artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[
                           'Fica', 'Churn']).plot(ax=ax_cm, cmap='Blues')
    ax_cm.set_title(f"Matriz de Confusão (Threshold {threshold})")
    fig_cm.savefig("outputs/confusion_matrix.png")
    mlflow.log_artifact("outputs/confusion_matrix.png",
                        artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: CLASSIFICATION REPORT ---
    report_path = "outputs/classification_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Classification Report - Threshold: {threshold}\n")
        f.write("-" * 50 + "\n")
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact(report_path, artifact_path="diagnostics")

    # --- ARTEFATO 4: PREPROCESSADOR E METADADOS ---
    joblib.dump(preprocessador, "outputs/preprocessador.pkl")
    mlflow.log_artifact("outputs/preprocessador.pkl",
                        artifact_path="preprocessor")

    with open("outputs/feature_names.json", "w") as f:
        json.dump(list(X_train_df.columns), f)
    mlflow.log_artifact("outputs/feature_names.json", artifact_path="data")

    # --- SALVAR MODELO COM SIGNATURE ---
    signature = infer_signature(X_test_df, y_pred)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_test_df.head(5)
    )

    # --- MODEL CARD (SE EXISTIR) ---
    if os.path.exists("docs/model-card.md"):
        mlflow.log_artifact("docs/model-card.md", artifact_path="model_card")

print(f"Finalizado! AUC-ROC: {roc_auc:.4f} | Custo Total: R${total_cost:.2f}")


c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Python311\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-in

Finalizado! AUC-ROC: 0.8435 | Custo Total: R$50440.00


# --------------- DUMMY CLASSIFIER ---------------

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
import os
import json
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, accuracy_score,
                             f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, auc, classification_report)
from mlflow.models.signature import infer_signature

# --- 1. CONFIGURAÇÕES E CUSTOS (Alinhado com o Baseline) ---
SEED = 20
test_size = 0.2
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325
THRESHOLD = 0.30             # Threshold definido pelo usuário

os.makedirs("outputs_dummy", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(), X_train.select_dtypes(
            include=['int64', 'float64']).columns)
    ]
)

# Transformação para o fit
X_train_transformed = preprocessador.fit_transform(X_train)
X_test_transformed = preprocessador.transform(X_test)
feature_names = preprocessador.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

# Dataset para o MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset_mlflow = mlflow.data.from_pandas(
    df, source=csv_path, name="Churn_Analysis_Base")

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Dummy_Classifier_baseline_v1"):
    mlflow.set_tags({"phase": "baseline", "model_type": "dummy_classifier"})
    # Modelo Dummy
    modelo_dummy = DummyClassifier(strategy='stratified', random_state=SEED)
    mlflow.log_input(dataset, context="training")
    modelo_dummy.fit(X_train_df, y_train)

    # Predições (Dummy também usa threshold de 0.5 por padrão na probabilidade)
    y_probs = modelo_dummy.predict_proba(X_test_df)[:, 1]
    y_pred = (y_probs >= THRESHOLD).astype(int)

    # Cálculo de Métricas Técnicas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    # Cálculo de Custo de Negócio
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = (fp * COST_FP) + (fn * COST_FN)

    # Log de Parâmetros e Tags
    mlflow.set_tags({"model_type": "baseline_dummy",
                    "framework": "scikit-learn"})
    mlflow.log_params({
        "strategy": "stratified",
        "model_type": "DummyClassifier",
        "stratify": True,
        "test_size": test_size,
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "random_state": SEED,
        "cost_fp": COST_FP,
        "cost_fn": COST_FN,
        'threshold': THRESHOLD
    })

    # Log de Métricas
    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.total_cost": total_cost
    })

    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='gray', lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC - Dummy Classifier")
    ax_roc.legend()
    fig_roc.savefig("outputs_dummy/roc_curve_dummy.png")
    mlflow.log_artifact("outputs_dummy/roc_curve_dummy.png",
                        artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[
                           'Fica', 'Churn']).plot(ax=ax_cm, cmap='Greys')
    ax_cm.set_title('Matriz de Confusão - Dummy Classifier')
    fig_cm.savefig("outputs_dummy/confusion_matrix_dummy.png")
    mlflow.log_artifact(
        "outputs_dummy/confusion_matrix_dummy.png", artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: CLASSIFICATION REPORT ---
    report_path = "outputs_dummy/classification_report_dummy.txt"
    with open(report_path, "w") as f:
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact(report_path, artifact_path="diagnostics")

    # --- ARTEFATO 4: PREPROCESSADOR E DATA ---
    joblib.dump(preprocessador, "outputs_dummy/preprocessador.pkl")
    mlflow.log_artifact("outputs_dummy/preprocessador.pkl",
                        artifact_path="preprocessor")

    with open("outputs_dummy/feature_names.json", "w") as f:
        json.dump(list(feature_names), f)
    mlflow.log_artifact("outputs_dummy/feature_names.json",
                        artifact_path="data")

    # --- SALVAR MODELO COM SIGNATURE ---
    signature = infer_signature(X_test_df, y_pred)
    mlflow.sklearn.log_model(modelo_dummy, "model", signature=signature)

    print(f"===== Dummy Classifier Gravado! Custo Negócio: R$ {
          total_cost:.2f} =====")


c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/04/24 17:22:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 17:22:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


===== Dummy Classifier Gravado! Custo Negócio: R$ 109525.00 =====


# --------------- ÁRVORE DE DECISÃO ---------------

In [43]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature

# --- 1. CONFIGURAÇÕES E CUSTOS ---
SEED = 42
test_size = 0.2
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325
max_depth = 5 

os.makedirs("outputs_dt", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=SEED, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(), X_train.select_dtypes(include=['int64', 'float64']).columns)
    ]
)

X_train_df = pd.DataFrame(preprocessador.fit_transform(X_train), columns=preprocessador.get_feature_names_out())
X_test_df = pd.DataFrame(preprocessador.transform(X_test), columns=preprocessador.get_feature_names_out())

# Dataset para o MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset_mlflow = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Decision_Tree_Baseline_v1"):
    mlflow.log_input(dataset, context="training")
    model_dt = DecisionTreeClassifier(max_depth=max_depth, random_state=SEED)
    model_dt.fit(X_train_df, y_train)

    # Predições com Threshold Ajustado (0.30) para diminuir Falso Negativo
    y_probs = model_dt.predict_proba(X_test_df)[:, 1]
    threshold = 0.30
    y_pred = (y_probs >= threshold).astype(int)

    # Cálculos de Métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)
    
    cm = confusion_matrix(y_test, y_pred)
    total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

    # Log de Tags e Parâmetros
    mlflow.set_tags({"model_type": "decision_tree", "framework": "scikit-learn", "phase": "baseline"})
    mlflow.log_params({
        "model_type": "DecisionTreeClassifier",
        "random_state": random_state,
        "test_size":test_size,
        "stratify": True,
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "max_depth": max_depth,
        "threshold": threshold,
        "cost_fn_weight": 5
    })

    # Log de Métricas
    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.total_cost": total_cost
    })

    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='seagreen', lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC - Decision Tree")
    ax_roc.legend()
    fig_roc.savefig("outputs_dt/roc_curve_dt.png")
    mlflow.log_artifact("outputs_dt/roc_curve_dt.png", artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fica', 'Churn']).plot(ax=ax_cm, cmap='Greens')
    ax_cm.set_title(f'Matriz de Confusão DT (Threshold {threshold})')
    fig_cm.savefig("outputs_dt/confusion_matrix_dt.png")
    mlflow.log_artifact("outputs_dt/confusion_matrix_dt.png", artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: FEATURE IMPORTANCE (O diferencial da Árvore!) ---
    importances = model_dt.feature_importances_
    fi_df = pd.DataFrame({'feature': X_train_df.columns, 'importance': importances}).sort_values(by='importance', ascending=False)
    
    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    top_fi = fi_df.head(15)
    ax_fi.barh(top_fi['feature'][::-1], top_fi['importance'][::-1], color='seagreen')
    ax_fi.set_title("Top 15 Features Mais Decisivas")
    fig_fi.savefig("outputs_dt/feature_importance.png")
    mlflow.log_artifact("outputs_dt/feature_importance.png", artifact_path="diagnostics")
    plt.close(fig_fi)

    # --- ARTEFATO 4: CLASSIFICATION REPORT ---
    with open("outputs_dt/classification_report.txt", "w") as f:
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact("outputs_dt/classification_report.txt", artifact_path="diagnostics")

    # --- ARTEFATO 5: PREPROCESSADOR E DATA ---
    joblib.dump(preprocessador, "outputs_dt/preprocessador.pkl")
    mlflow.log_artifact("outputs_dt/preprocessador.pkl", artifact_path="preprocessor")
    
    with open("outputs_dt/feature_names.json", "w") as f:
        json.dump(list(X_train_df.columns), f)
    mlflow.log_artifact("outputs_dt/feature_names.json", artifact_path="data")

    # --- SALVAR MODELO COM SIGNATURE ---
    signature = infer_signature(X_test_df, y_pred)
    mlflow.sklearn.log_model(model_dt, "model", signature=signature)

print(f"Árvore de Decisão Gravada! AUC-ROC: {roc_auc:.4f} | Custo: R$ {total_cost:.2f}")

c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/04/24 17:23:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 17:23:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Árvore de Decisão Gravada! AUC-ROC: 0.8335 | Custo: R$ 43615.00


# --------------- RANDOM FOREST ---------------

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature

# --- 1. CONFIGURAÇÕES E CUSTOS ---
SEED = 42
test_size = 0.2
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325
n_estimators = 100
max_depth = 10

os.makedirs("outputs_rf", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=SEED, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(), X_train.select_dtypes(
            include=['int64', 'float64']).columns)
    ]
)

X_train_df = pd.DataFrame(preprocessador.fit_transform(
    X_train), columns=preprocessador.get_feature_names_out())
X_test_df = pd.DataFrame(preprocessador.transform(
    X_test), columns=preprocessador.get_feature_names_out())

# Dataset para o MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset_mlflow = mlflow.data.from_pandas(
    df, source=csv_path, name="Churn_Analysis_Base")

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Random_Forest_Baseline_v1"):
    mlflow.log_input(dataset, context="training")
    # Definição do Modelo
    model_rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=SEED,
        n_jobs=-1
    )
    model_rf.fit(X_train_df, y_train)

    # Predições com Threshold de 0.30 (Consistência com os outros modelos)
    y_probs = model_rf.predict_proba(X_test_df)[:, 1]
    threshold = 0.30
    y_pred = (y_probs >= threshold).astype(int)

    # Cálculos de Métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    cm = confusion_matrix(y_test, y_pred)
    total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

    # Log de Tags e Parâmetros
    mlflow.set_tags({"model_type": "random_forest",
                    "framework": "scikit-learn", "phase": "baseline"})
    mlflow.log_params({
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "threshold": threshold,
        "cost_fn_weight": 5,
        "model_type": "RandomForestClassifier",
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # Log de Métricas
    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.total_cost": total_cost
    })

    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='rebeccapurple',
                lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC - Random Forest")
    ax_roc.legend()
    fig_roc.savefig("outputs_rf/roc_curve_rf.png")
    mlflow.log_artifact("outputs_rf/roc_curve_rf.png",
                        artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[
                           'Fica', 'Churn']).plot(ax=ax_cm, cmap='Purples')
    ax_cm.set_title(f'Matriz de Confusão RF (Threshold {threshold})')
    fig_cm.savefig("outputs_rf/confusion_matrix_rf.png")
    mlflow.log_artifact("outputs_rf/confusion_matrix_rf.png",
                        artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: FEATURE IMPORTANCE ---
    importances = model_rf.feature_importances_
    fi_df = pd.DataFrame({'feature': X_train_df.columns, 'importance': importances}).sort_values(
        by='importance', ascending=False)

    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    top_fi = fi_df.head(15)
    ax_fi.barh(top_fi['feature'][::-1], top_fi['importance']
               [::-1], color='rebeccapurple')
    ax_fi.set_title("Top 15 Features - Random Forest")
    fig_fi.savefig("outputs_rf/feature_importance_rf.png")
    mlflow.log_artifact("outputs_rf/feature_importance_rf.png",
                        artifact_path="diagnostics")
    plt.close(fig_fi)

    # --- ARTEFATO 4: CLASSIFICATION REPORT ---
    with open("outputs_rf/classification_report.txt", "w") as f:
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact("outputs_rf/classification_report.txt",
                        artifact_path="diagnostics")

    # --- ARTEFATO 5: PREPROCESSADOR E DATA ---
    joblib.dump(preprocessador, "outputs_rf/preprocessador.pkl")
    mlflow.log_artifact("outputs_rf/preprocessador.pkl",
                        artifact_path="preprocessor")

    with open("outputs_rf/feature_names.json", "w") as f:
        json.dump(list(X_train_df.columns), f)
    mlflow.log_artifact("outputs_rf/feature_names.json", artifact_path="data")

    # --- SALVAR MODELO COM SIGNATURE ---
    signature = infer_signature(X_test_df, y_pred)
    mlflow.sklearn.log_model(model_rf, "model", signature=signature)

print(
    f"Random Forest Gravado! AUC-ROC: {roc_auc:.4f} | Custo Total: R$ {total_cost:.2f}")


c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
2026/04/24 17:23:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/24 17:23:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Gravado! AUC-ROC: 0.8384 | Custo Total: R$ 41795.00


========================= COMPARACAO DE METRICAS ENTRE MODELOS + ANALISE DE CUSTOS ==========================

In [52]:
import mlflow
import pandas as pd

experiment_name = "Projeto_Churn_Tech_Challenge"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment:
    # Busca as runs
    df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    
    # --- PASSO 1: LIMPEZA AGRESSIVA DE PREFIXOS ---
    # Isso remove metrics., params., tags. e também o test. que costuma sobrar
    df_runs.columns = [c.replace("metrics.test.", "").replace("metrics.", "")
                        .replace("params.", "").replace("tags.mlflow.", "")
                        .replace("test.", "") for c in df_runs.columns]

    # --- PASSO 2: REMOVER DUPLICADAS ---
    df_runs = df_runs.loc[:, ~df_runs.columns.duplicated()].copy()

    # --- PASSO 3: DEBUG (Para você ver se as colunas estão lá) ---
    print("Colunas encontradas:", df_runs.columns.tolist())

    # --- PASSO 4: FILTRAGEM SEGURA ---
    # Verificamos quais das colunas que queremos REALMENTE existem no DF
    cols_desejadas = ["model_type", "accuracy", "recall", "precision", "roc_auc", "business.total_cost"]
    cols_que_existem = [c for c in cols_desejadas if c in df_runs.columns]
    
    # Só fazemos o dropna nas colunas que de fato existem
    df_clean = df_runs.dropna(subset=cols_que_existem).copy()

    if df_clean.empty:
        print("AVISO: O DataFrame continua vazio após o dropna. Verifique se os nomes acima estão corretos.")
    else:
        # --- PASSO 5: LÓGICA DO LEADERBOARD ---
        # Ordenamos pelo custo total (menor é melhor)
        leaderboard = df_clean.sort_values("business.total_cost", ascending=True).drop_duplicates("model_type")
        
        # Selecionamos apenas o que existe
        leaderboard = leaderboard[cols_que_existem]

        # Renomeação dinâmica (para evitar o erro de Length Mismatch)
        nomes_map = {
            "model_type": "Modelo",
            "accuracy": "Acurácia",
            "recall": "Recall",
            "precision": "Precisão",
            "roc_auc": "AUC-ROC",
            "business.total_cost": "Custo Total (Impacto)"
        }
        
        # Renomeia apenas as que existem na lista
        leaderboard.rename(columns=nomes_map, inplace=True)

        # Formatação de Moeda (se a coluna existir)
        col_custo = "Custo Total (Impacto)"
        if col_custo in leaderboard.columns:
            leaderboard[col_custo] = leaderboard[col_custo].apply(lambda x: f"R$ {float(x):,.2f}")

        print("===== LEADERBOARD FINAL: IMPACTO NO NEGÓCIO =====")
        display(leaderboard)
else:
    print("Experimento não encontrado.")

Colunas encontradas: ['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'precision', 'recall', 'accuracy', 'roc_auc', 'business.total_cost', 'f1_score', 'val_loss', 'train_loss', 'total_churn_count', 'avg_revenue_lost_per_client', 'churn_rate_percent', 'overfitting', 'accuracy_train', 'accuracy_test', 'cv_recall', 'cv_precision', 'cv_accuracy', 'cv_f1', 'train_samples', 'max_depth', 'test_samples', 'cost_fn_weight', 'random_state', 'n_estimators', 'model_type', 'threshold', 'stratify', 'test_size', 'cost_fn', 'cost_fp', 'strategy', 'max_iter', 'ticket_medio', 'lr', 'batch_size', 'dropout', 'layers', 'pos_weight', 'critical_product', 'critical_contract', 'optimizer', 'architecture', 'early_stopping_patience', 'user', 'tags.phase', 'source.type', 'tags.framework', 'source.name', 'tags.model_type', 'runName', 'source.git.commit']
===== LEADERBOARD FINAL: IMPACTO NO NEGÓCIO =====


,Modelo,Acurácia,Recall,Precisão,AUC-ROC,Custo Total (Impacto)
0,RandomForestClassifier,0.756565,0.799465,0.527337,0.838352,"R$ 41,795.00"
10,DecisionTreeClassifier,0.745209,0.791444,0.512998,0.833534,"R$ 43,615.00"
14,LogisticRegression,0.755855,0.748663,0.528302,0.843509,"R$ 46,800.00"
3,DummyClassifier,0.604684,0.245989,0.250681,0.490144,"R$ 109,525.00"
